For the EEG Experiment the design matrix needs to:
- contain double the amount of neutral cues compared to the other 2
- use only 4 images, either (_00 or _01)



For the EEG Experiment we need to come up with a scheme for the triggers 
We will have a 2 x 3 x 4 design (Masking x Expectation x images) - this gives me 24 unique triggers 
The triggers with Brain products can vary from 1 x 250 


- images: 0-1-2-3 \\
- mask: + 10-20 \\
- expectation: decimals, 100s, 200s (neutral, expected, unexpected)\\


11 -> image 1: early-mask, neutral \
111-> image 1: early-mask, expected \
211-> image 1: early-mask, unexpected \

21 -> image 1: late-mask, neutral \
121-> image 1: late-mask, expected \
221-> image 1: late-mask, unexpected \


In [30]:
cue_data = create_cue_dynam()
stim_path = "/projects/crunchie/boyanova/EEG_Things/Mask_ExpAtt_EEG/01_EEG_Task/stimuli"

pick_images = "_01"
long_isi = 0.100


In [31]:
for key in data.keys():
    print(f"{key}:{len(data[key])}")

target_id:320
target:320
expectation:320
mask_ISI:320
cue:320
target_name:320
target_cat:320


In [26]:
cue_data = create_cue_dynam()
cue_data

{'cue_names': ['Sea animal', 'Water vessel', 'Neutral'],
 'cue_letters': ['SA', 'WV', 'X'],
 'cue_color': [[0.4, 0.6, 1.0], [-0.4, -0.2, 0.4], [-1, -1, -1]],
 'cue_highProb_cats': [['dolphin', 'whale'],
  ['speedboat', 'submarine'],
  ['dolphin', 'whale', 'speedboat', 'submarine']],
 'cue_lowProb_cats': [['speedboat', 'submarine'],
  ['dolphin', 'whale'],
  ['dolphin', 'whale', 'speedboat', 'submarine']],
 'cue_highProb': [[0.35, 0.35], [0.35, 0.35], [0.25, 0.25, 0.25, 0.25]],
 'cue_lowProb': [[0.15, 0.15], [0.15, 0.15], [0.25, 0.25, 0.25, 0.25]],
 'high_prob_trials': [array([14., 14.]),
  array([14., 14.]),
  array([20., 20., 20., 20.])],
 'low_prob_trials': [array([6., 6.]),
  array([6., 6.]),
  array([20., 20., 20., 20.])]}

In [29]:
from collections import Counter


Counter(data["cue"]).keys() # equals to list(set(words))
Counter(data["cue"]).values() # counts the elements' frequency

dict_values([80, 80, 320])

In [65]:
import numpy as np
import os
import random
import pandas as pd 

def create_cue_dynam(highProb=0.7, lowProb=0.3, neutral=1.0, trials_per_cue=40):
    
    # double the amount for EEG 
    trial_per_neutral = 2 * trials_per_cue
    
    cue_data = {"cue_names": [".\\cues\\Sea_Animal.png",  ".\\cues\\Water_Vehicle.png",  ".\\cues\\Neutral.png"],
                "cue_highProb_cats": [["dolphin", "whale"], ["speedboat", "submarine"], 
                                    ["dolphin", "whale", "speedboat", "submarine"]],
                
                "cue_lowProb_cats": [["speedboat", "submarine"], ["dolphin", "whale"],
                                    ["dolphin", "whale", "speedboat", "submarine"]]}

    cue_data["cue_highProb"] = []
    cue_data["cue_lowProb"] = []
    
    for cue_id, cue in enumerate(cue_data["cue_names"]):
        if cue != ".\\cues\\Neutral.png":
            cue_data["cue_highProb"].append([np.round(highProb / len(cue_data["cue_highProb_cats"][cue_id]), 2)] * len(cue_data["cue_highProb_cats"][cue_id]))
            cue_data["cue_lowProb"].append([np.round(lowProb / len(cue_data["cue_lowProb_cats"][cue_id]), 2)] * len(cue_data["cue_lowProb_cats"][cue_id]))
        
        else:
            cue_data["cue_highProb"].append([np.round(neutral / len(cue_data["cue_highProb_cats"][cue_id]), 3)] * len(cue_data["cue_highProb_cats"][cue_id]))
            cue_data["cue_lowProb"].append([np.round(neutral / len(cue_data["cue_lowProb_cats"][cue_id]), 3)] * len(cue_data["cue_highProb_cats"][cue_id]))
            
    cue_data["high_prob_trials"] = [
    np.array(x) * (trial_per_neutral if cue_data["cue_names"][idx] == ".\\cues\\Neutral.png" else trials_per_cue)
    for idx, x in enumerate(cue_data["cue_highProb"])]
    
    cue_data["low_prob_trials"] =  [
    np.array(x) * (trial_per_neutral if cue_data["cue_names"][idx] == ".\\cues\\Neutral.png" else trials_per_cue)
    for idx, x in enumerate(cue_data["cue_lowProb"])]
    
    return cue_data

def assign_trigger(row, late=0.100):
    
    # ---- MASK ----
    if row['mask_ISI'] == 0.017:
        mask_code = 10
    elif row['mask_ISI'] == late:
        mask_code = 20
    else:
        raise ValueError("Unknown mask type")
    
    # ---- IMAGE (side dependent) ----
    image_code = row['image_index']
    base_code = mask_code + image_code
    
    # ---- EXPECTATION ----
    if row['expectation'] == 'neutral':
        exp_code = 0
    elif row['expectation'] == 'expected':
        exp_code = 100
    elif row['expectation'] == 'unexpected':
        exp_code = 200
    else:
        raise ValueError("Unknown expectation")
    
    return base_code + exp_code

def allocate_catch_trials(N, p=0.3, k=3, alpha=0.66):
    C = int(N * p)
    
    # ensure divisible by (k-1)*2 if needed
    while C % (k-1) != 0:
        C -= 1
    
    main = int(round(alpha * C))
    
    # enforce even split
    remainder = C - main
    other = remainder // (k - 1)
    
    # adjust if rounding broke divisibility
    main = C - other * (k - 1)
    
    return main, [other] * (k - 1)

def create_changes(list_length, prec):
    # Number of instances to set as True (2%)
    num_true = int(list_length * prec)

    # Create lists of False values
    catch = [False] * list_length

    # Randomly select indices for fixing and imaging channels
    catch_indices = random.sample(range(list_length), num_true)

    for idx in catch_indices:
        catch[idx] = True
    
    return np.array(catch)

def build_constrained_order(df, seed=None, max_unexpected_run=1):
    rng = np.random.default_rng(seed)

    remaining = df.copy()
    ordered_rows = []

    last_target = None
    unexpected_run = 0

    while len(remaining) > 0:

        # valid candidates mask
        valid_mask = np.ones(len(remaining), dtype=bool)

        # Rule 1 — no same target twice
        if last_target is not None:
            valid_mask &= (remaining["target"].values != last_target)

        # Rule 2 — max unexpected run
        if unexpected_run >= max_unexpected_run:
            valid_mask &= (remaining["expectation"].values != "unexpected")

        valid = remaining[valid_mask]

        # if dead end → restart whole sequence
        if len(valid) == 0:
            return build_constrained_order(df, seed=rng.integers(0,1e9))

        # pick random valid row
        choice_idx = rng.integers(len(valid))
        row = valid.iloc[choice_idx]

        ordered_rows.append(row)

        # update state
        last_target = row["target"]
        if row["expectation"] == "unexpected":
            unexpected_run += 1
        else:
            unexpected_run = 0

        # remove selected row
        remaining = remaining.drop(valid.index[choice_idx])

    return pd.DataFrame(ordered_rows).reset_index(drop=True)

def create_block_trials(stim_path, cue_data, random_seed, long_isi=0.1, pick_images="_01", identity_catch=0.1): 
    categories = os.listdir(stim_path)
    stimuli = []
    for cat in categories:
        cat_path = os.path.join(stim_path, cat)
        files = os.listdir(cat_path)
        stimuli.extend([f".\\stimuli\\{cat}\\{x}" for x in files])

    # Filter stims
    stimuli = np.array(stimuli)
    stimuli = np.array([x for x in stimuli if pick_images in x ])
    stimuli = stimuli[np.argsort(stimuli)]

    data = {"target_id": [],
            "target": [],
            "expectation": [],
            "mask_ISI": [],
            "cue": [],
            "target_name": [],
            "target_cat": []}

    mask_type = [0.017, long_isi]
    for cue_id, cue in enumerate(cue_data["cue_names"]):
        high_cats = np.array(cue_data["cue_highProb_cats"][cue_id])
        low_cats = np.array(cue_data["cue_lowProb_cats"][cue_id])

        # since the neutral category has all 4 images there is no need to repeat it twice
        if cue != ".\\cues\\Neutral.png":
            # This loop handles only unexpexted cases
            for i, l_cat in enumerate(low_cats):
                l_cat_stim = stimuli[np.char.count(stimuli, l_cat) > 0]
                targets = l_cat_stim[np.char.count(l_cat_stim, "mask") == 0]
                target_ids = np.arange(len(stimuli))[np.isin(stimuli, targets)]
            

                for mask in mask_type:
                
                    h_trials = int(cue_data["low_prob_trials"][cue_id][i])
                
                    data["target_id"].extend(np.repeat(target_ids , h_trials))
                    data["target"].extend(np.repeat(targets, h_trials))
                    
                    target_names =[x.split("\\")[-1] for x in targets]
                    target_categories = [x.split("_")[0] for x in target_names]               

                    data["target_name"].extend(np.repeat(target_names , h_trials))
                    data["target_cat"].extend(np.repeat(target_categories , h_trials))
                    
                
                    data["expectation"].extend(["unexpected"] * h_trials)

                        
                    data["mask_ISI"].extend([mask] * h_trials)
                    data["cue"].extend([cue] * h_trials)
        
        # This loop handles expexted and neutral cases         
        for i, h_cat in enumerate(high_cats):
            h_cat_stim = stimuli[np.char.count(stimuli, h_cat) > 0]
            targets = h_cat_stim[np.char.count(h_cat_stim, "mask") == 0]
            target_ids = np.arange(len(stimuli))[np.isin(stimuli, targets)]

            for mask in mask_type:
                h_trials = int(cue_data["high_prob_trials"][cue_id][i])
                
                data["target_id"].extend(np.repeat(target_ids , h_trials))
                data["target"].extend(np.repeat(targets , h_trials))
                
                target_names =[x.split("\\")[-1] for x in targets]
                target_categories = [x.split("_")[0] for x in target_names]
                

                data["target_name"].extend(np.repeat(target_names , h_trials))
                data["target_cat"].extend(np.repeat(target_categories , h_trials))
                
                
                if cue != ".\\cues\\Neutral.png":
                    data["expectation"].extend(["expected"] * h_trials)
                else:
                    data["expectation"].extend(["neutral"]* h_trials)
                    
                data["mask_ISI"].extend([mask] * h_trials)
                data["cue"].extend([cue] * h_trials)
            
    mapping = {0: 1,
               3: 2,
               1: 3,
               2: 4}
    
    df = pd.DataFrame(data)
    df['image_index'] = df['target_id'].map(mapping)
    df['trigger'] = df.apply(assign_trigger, axis=1)
    df["identity_catch"] = create_changes(len(df), identity_catch)
    df = build_constrained_order(df, seed=random_seed)
    
    return df, stimuli

In [66]:
cue_data = create_cue_dynam()
stim_path = "/projects/crunchie/boyanova/EEG_Things/Mask_ExpAtt_EEG/01_EEG_Task/stimuli"
trials, stimss = create_block_trials(stim_path, cue_data, random_seed=19)

In [70]:
trials.head()

,target_id,target,expectation,mask_ISI,cue,target_name,target_cat,image_index,trigger,identity_catch
0,0,.\stimuli\dolphin\dolphin_01.jpg,neutral,0.100,.\cues\Neutral.png,dolphin_01.jpg,dolphin,1,21,False
1,2,.\stimuli\submarine\submarine_01.jpg,expected,0.017,.\cues\Water_Vehicle.png,submarine_01.jpg,submarine,4,114,False
2,3,.\stimuli\whale\whale_01.jpg,unexpected,0.017,.\cues\Water_Vehicle.png,whale_01.jpg,whale,2,212,False
3,2,.\stimuli\submarine\submarine_01.jpg,neutral,0.100,.\cues\Neutral.png,submarine_01.jpg,submarine,4,24,False
4,3,.\stimuli\whale\whale_01.jpg,unexpected,0.100,.\cues\Water_Vehicle.png,whale_01.jpg,whale,2,222,False


In [43]:
unique_combos = trials[['trigger', 'target_name', 'target_id']].drop_duplicates()

In [68]:
unique_combos.sort_values("trigger")

,trigger,target_name,target_id
15,11,dolphin_01.jpg,0
18,12,whale_01.jpg,3
11,13,speedboat_01.jpg,1
6,14,submarine_01.jpg,2
0,21,dolphin_01.jpg,0
13,22,whale_01.jpg,3
32,23,speedboat_01.jpg,1
3,24,submarine_01.jpg,2
7,111,dolphin_01.jpg,0
10,112,whale_01.jpg,3


In [1]:
import os
from PIL import Image

root_dir = "/projects/crunchie/boyanova/EEG_Things/Mask_ExpAtt_EEG/target_stimuli"
target_size = (500, 500)

# Image extensions to process
valid_exts = (".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp")

for root, dirs, files in os.walk(root_dir):
    for file in files:
        if file.lower().endswith(valid_exts):
            img_path = os.path.join(root, file)

            try:
                with Image.open(img_path) as img:
                    img = img.convert("RGB")  # safe for consistency
                    img_resized = img.resize(target_size, Image.LANCZOS)
                    img_resized.save(img_path)

            except Exception as e:
                print(f"Failed to process {img_path}: {e}")
